In [1]:
import os
import torch
from dotenv import load_dotenv
from crewai import LLM, Agent, Task, Crew, Process
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from crewai.tools import tool

load_dotenv()

# Set device
device = "cuda" if torch.cuda.is_available() else "cpu"

# Initialize LLMs (Using Groq as per your LegalRAG2 setup)
legal_llm = LLM(model="groq/llama-3.3-70b-versatile")
research_llm = LLM(model="groq/llama-3.1-8b-instant")

# Set dummy key to satisfy any internal Pydantic checks (Method from Lawglance)
os.environ["OPENAI_API_KEY"] = "NA"

In [2]:
# 1. Initialize Embeddings
embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-base-en-v1.5",
    model_kwargs={'device': device}
)

# 2. Connect to your existing Vector Database
# Update this path to your actual vector_database folder
db_path = "C:/Users/user/Desktop/RAG Projects/Legal RAG 1/vector_database"
vector_store = Chroma(persist_directory=db_path, embedding_function=embeddings)

# 3. Create the Custom Tool function
@tool("legal_research_tool")
def legal_research_tool(query: str):
    """
    Searches the official legal vector database. 
    Use this tool to find specific statutes, constitutional articles, and case law relevant to the user's query.
    """
    # Using similarity search logic from your chains.py
    docs = vector_store.similarity_search(query, k=5)
    
    # Combine results into a single string for the agent to read
    return "\n\n".join([f"Source Content: {doc.page_content}" for doc in docs])

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-base-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [3]:
# Agent 1: The Researcher
researcher = Agent(
    role='Legal Researcher',
    goal='Retrieve exact legal provisions and citations related to {user_query}',
    backstory='Expert in Indian Constitutional law and criminal statutes. Skilled at finding the needle in the legal haystack.',
    tools=[legal_research_tool],
    llm=research_llm,
    verbose=True
)

# Agent 2: The Strategist
strategist = Agent(
    role='Legal Strategist',
    goal='Analyze the findings from the researcher and determine the legal hierarchy (e.g., Central vs. State law)',
    backstory='Senior legal counsel specializing in jurisdictional conflicts and constitutional interpretation.',
    llm=legal_llm,
    verbose=True
)

# Agent 3: The Advisor
advisor = Agent(
    role='Legal Advisor',
    goal='Synthesize a final, easy-to-understand response for the user based on the strategist\'s analysis',
    backstory='A compassionate legal advisor who translates complex legalese into actionable advice for citizens.',
    llm=legal_llm,
    verbose=True
)

In [4]:
research_task = Task(
    description='Search the database for all legal articles or sections relevant to: {user_query}. Provide direct quotes.',
    expected_output='A list of specific legal sections and their content.',
    agent=researcher
)

strategy_task = Task(
    description='Based on the research, explain the legal implications. If there is a conflict between laws, explain which one prevails.',
    expected_output='A strategic analysis of the legal hierarchy and applicable rules.',
    agent=strategist
)

advisory_task = Task(
    description='Summarize the strategy into a final answer for the user. Be concise and professional.',
    expected_output='A final response that directly answers the user\'s original question.',
    agent=advisor
)

In [5]:
legal_crew = Crew(
    agents=[researcher, strategist, advisor],
    tasks=[research_task, strategy_task, advisory_task],
    process=Process.sequential,
    memory=False, # This is CRITICAL to avoid the Pydantic/Embedder error
    verbose=True
)

In [6]:
# Execute
user_input = "If a state law conflicts with a central law in India, which one prevails?"
result = legal_crew.kickoff(inputs={"user_query": user_input})

print("\n" + "="*30)
print("FINAL LEGAL ADVICE:")
print(result.raw)

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name:                                                                                                          │
│  crew                                                                                                           │
│  ID:                                                                                                            │
│  f0593dc9-fdd5-4c1c-97e9-f38e700906b3                                                                           │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Search the database for all legal articles or sections relevant to: If a state law conflicts with a      │
│  central law in India, which one prevails?. Provide direct quotes.                                              │
│  ID: 0e199204-0bca-45df-89b7-a700ad6f6c56                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Legal Researcher                                                                                        │
│                                                                                                                 │
│  Task: Search the database for all legal articles or sections relevant to: If a state law conflicts with a      │
│  central law in India, which one prevails?. Provide direct quotes.                                              │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────── 🔧 Tool Execution Started (#1) ─────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: legal_research_tool                                                                                      │
│  Args: {'query': 'In case of conflict between state law and central law in India, the central law prevails.'}   │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Tool legal_research_tool executed with result: ...


...

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name:                                                                                                          │
│  Summarize the strategy into a final answer for the user. Be concise and professional.                          │
│  Agent:                                                                                                         │
│  Legal Advisor                                                                                                  │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

[CrewAIEventsBus] Warning: Event pairing mismatch. 'crew_kickoff_completed' closed 'task_started' (expected 
'crew_kickoff_started')

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name:                                                                                                          │
│  crew                                                                                                           │
│  ID:                                                                                                            │
│  f0593dc9-fdd5-4c1c-97e9-f38e700906b3                                                                           │
│  Final Output: The Constitution of India, specifically Articles 245 and 254, provides a framework for           │
│  understanding the legal hierarchy and the resolution of conflicts between Central and State laws. Article 245  │
│  emphasizes the supremacy of the Constitution and the need for the executive power of the Union to ensure that  │
│  state governments operate in accordance with the constitutional provisions and guarantees. This article sets   │
│  the stage for understanding the distribution of power between the Central government and the states.           │
│                                                                                                                 │
│  In the event of a conflict between a Central law and a state law, Article 254(1) of the Constitution comes     │
│  into play. This article explicitly states that when a state law is repugnant to a Central law in regards to a  │
│  matter listed in the Concurrent List (List III of the Seventh Schedule), the Central law prevails. The         │
│  Concurrent List contains subjects over which both the Central government and the state governments have the    │
│  power to legislate. The prevalence of Central law over state law in such cases is a key aspect of the legal    │
│  hierarchy, as it ensures uniformity and consistency in the application of laws across the country on subjects  │
│  of concurrent jurisdiction.                                                                                    │
│                                                                                                                 │
│  However, it is crucial to note that the Constitution of India does not provide an absolute hierarchy where     │
│  the Central government's laws always prevail over those of the states. The prevalence of Central law is        │
│  specifically limited to cases of repugnancy concerning subjects in the Concurrent List. For matters falling    │
│  exclusively under the State List (List II), state laws govern, and there is no provision for Central laws to   │
│  override them unless the subject matter intersects with the Concurrent List or unless the state law            │
│  contravenes a fundamental right or a constitutional provision.                                                 │
│                                                                                                                 │
│  **Applicable Rules and Hierarchy**                                                                             │
│                                                                                                                 │
│  1. **Constitutional Supremacy**: The Constitution of India is the supreme law of the land. All laws, whether   │
│  enacted by the Central government or the state governments, must conform to the provisions of the              │
│  Constitution.                                                                                                  │
│                                                                                                                 │
│  2. **Central Laws vs. State Laws**: In cases of confl


FINAL LEGAL ADVICE:
The Constitution of India, specifically Articles 245 and 254, provides a framework for understanding the legal hierarchy and the resolution of conflicts between Central and State laws. Article 245 emphasizes the supremacy of the Constitution and the need for the executive power of the Union to ensure that state governments operate in accordance with the constitutional provisions and guarantees. This article sets the stage for understanding the distribution of power between the Central government and the states.

In the event of a conflict between a Central law and a state law, Article 254(1) of the Constitution comes into play. This article explicitly states that when a state law is repugnant to a Central law in regards to a matter listed in the Concurrent List (List III of the Seventh Schedule), the Central law prevails. The Concurrent List contains subjects over which both the Central government and the state governments have the power to legislate. The prevalenc

╭──────────────────────────────────────────────── Tracing Status ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing is disabled.                                                                                     │
│                                                                                                                 │
│  To enable tracing, do any one of these:                                                                        │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

In [7]:
# Question 1
user_input = "I was caught jumping a red light and the police officer is asking for a 5,000 rupee fine. Is this amount correct?"
result = legal_crew.kickoff(inputs={"user_query": user_input})

# Verification Code
print(f"Evaluation: {'PASS' if 'Motor Vehicles Act' in result.raw else 'FAIL'}")
print(f"Reasoning: Check if the AI corrected the fine amount based on the database.")

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name:                                                                                                          │
│  crew                                                                                                           │
│  ID:                                                                                                            │
│  f0593dc9-fdd5-4c1c-97e9-f38e700906b3                                                                           │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Search the database for all legal articles or sections relevant to: I was caught jumping a red light     │
│  and the police officer is asking for a 5,000 rupee fine. Is this amount correct?. Provide direct quotes.       │
│  ID: 0e199204-0bca-45df-89b7-a700ad6f6c56                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Legal Researcher                                                                                        │
│                                                                                                                 │
│  Task: Search the database for all legal articles or sections relevant to: I was caught jumping a red light     │
│  and the police officer is asking for a 5,000 rupee fine. Is this amount correct?. Provide direct quotes.       │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────── 🔧 Tool Execution Started (#2) ─────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: legal_research_tool                                                                                      │
│  Args: {'query': 'Correct fine amount for jumping a red light in India.'}                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Tool legal_research_tool executed with result: ...


...

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name:                                                                                                          │
│  Summarize the strategy into a final answer for the user. Be concise and professional.                          │
│  Agent:                                                                                                         │
│  Legal Advisor                                                                                                  │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

[CrewAIEventsBus] Warning: Event pairing mismatch. 'crew_kickoff_completed' closed 'task_started' (expected 
'crew_kickoff_started')

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name:                                                                                                          │
│  crew                                                                                                           │
│  ID:                                                                                                            │
│  f0593dc9-fdd5-4c1c-97e9-f38e700906b3                                                                           │
│  Final Output: The Motor Vehicles Act, 1988, under Section 205 (1) mandates that a person committing a traffic  │
│  offence, punishable under Section 112 of the same Act, has to pay a fine of INR.100. Further, if a person      │
│  contravenes the provisions of Section 118(1), for the first offence, he is liable to a penalty of ₹5,000       │
│  under Section 205 (4), of the aforesaid Act.                                                                   │
│                                                                                                                 │
│  The Motor Vehicles Act, 1988 Section 205(1) states: 'When a person fails to comply with any provision of this  │
│  Act or of any rule or regulation or with any order made or direction given there under, he shall be            │
│  punishable with a penalty not exceeding one thousand rupees, and in the case of a continuing failure, with a   │
│  penalty which may extend to one hundred rupees for each day during which such failure continues.'              │
│                                                                                                                 │
│  The legal implications of the Motor Vehicles Act, 1988, particularly with regards to traffic offenses and      │
│  their corresponding penalties, are outlined in the Act itself. Section 205 of the Act provides a clear         │
│  framework for the penalties associated with various offenses committed under the Act. This analysis aims to    │
│  break down the legal implications and the hierarchy of rules applicable to traffic offenses as per the Motor   │
│  Vehicles Act, 1988.                                                                                            │
│                                                                                                                 │
│  **Understanding Section 205(1) and Its Implications**                                                          │
│                                                                                                                 │
│  Section 205(1) of the Motor Vehicles Act, 1988, mandates that any person who fails to comply with any          │
│  provision of the Act, or any rule, regulation, order, or direction made under the Act, shall be punishable     │
│  with a penalty. The maximum penalty for such non-compliance is set at INR 1,000. Additionally, if the failure  │
│  to comply continues over time, the individual may be liable for a further penalty of up to INR 100 for each    │
│  day the non-compliance continues. This provision is broad in scope, covering a wide range of potential         │
│  offenses under the Act, from minor infractions to more serious violations.                                     │
│                                                                                                                 │
│  **Specific Penalties for Certain Offenses**                                                                    │
│                                                                                                                 │
│  The context also mentions Section 112 of the Act, whi

Evaluation: PASS
Reasoning: Check if the AI corrected the fine amount based on the database.


╭──────────────────────────────────────────────── Tracing Status ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing is disabled.                                                                                     │
│                                                                                                                 │
│  To enable tracing, do any one of these:                                                                        │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

In [8]:
# Question 2
user_input = "I bought a phone last week and it stopped working. The shop owner says 'no returns'. What can I do?"
result = legal_crew.kickoff(inputs={"user_query": user_input})

# Verification Code
print(f"Evaluation: {'PASS' if 'Consumer Protection' in result.raw or 'Consumer' in result.raw else 'FAIL'}")

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name:                                                                                                          │
│  crew                                                                                                           │
│  ID:                                                                                                            │
│  f0593dc9-fdd5-4c1c-97e9-f38e700906b3                                                                           │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Search the database for all legal articles or sections relevant to: I bought a phone last week and it    │
│  stopped working. The shop owner says 'no returns'. What can I do?. Provide direct quotes.                      │
│  ID: 0e199204-0bca-45df-89b7-a700ad6f6c56                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Legal Researcher                                                                                        │
│                                                                                                                 │
│  Task: Search the database for all legal articles or sections relevant to: I bought a phone last week and it    │
│  stopped working. The shop owner says 'no returns'. What can I do?. Provide direct quotes.                      │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────── 🔧 Tool Execution Started (#3) ─────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: legal_research_tool                                                                                      │
│  Args: {'query': 'Consumer Rights Act, return policy for defective products in India.'}                         │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Tool legal_research_tool executed with result: ...


...

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name:                                                                                                          │
│  Summarize the strategy into a final answer for the user. Be concise and professional.                          │
│  Agent:                                                                                                         │
│  Legal Advisor                                                                                                  │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

[CrewAIEventsBus] Warning: Event pairing mismatch. 'crew_kickoff_completed' closed 'task_started' (expected 
'crew_kickoff_started')

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name:                                                                                                          │
│  crew                                                                                                           │
│  ID:                                                                                                            │
│  f0593dc9-fdd5-4c1c-97e9-f38e700906b3                                                                           │
│  Final Output: The Consumer Protection Act, 1986, under Section 12, provides for the right to seek refund of    │
│  the consideration, i.e., the price, paid by the consumer for the defective product or service.                 │
│                                                                                                                 │
│  Section 14 of the Consumer Protection Act, 1986, deals with the liability for defects in goods in case of      │
│  sale to the consumer. According to this section, any goods manufactured or sold by the supplier, are           │
│  warranted to be free from defects, for a period of one year from the date of sale and the supplier is liable   │
│  for replacing or repairing the defective goods during this period.                                             │
│                                                                                                                 │
│  Section 15 of the Consumer Protection Act, 1986, deals with action for redressal of grievances. The consumer   │
│  can approach the District Forum under Section 11, if he is not satisfied with the decision of the shop owner.  │
│  The District Forum under Section 11 can pass an order, requiring the shop owner to replace or repair the       │
│  defective product or return the consideration, if it is no longer required after sale.                         │
│                                                                                                                 │
│  The Consumer Protection Act, 1986, under Article 7, provides that the supplier shall be liable for any defect  │
│  in the goods or services which is supplied to the buyer. However, the supplier is not liable if the buyer has  │
│  tampered with the product after sale.                                                                          │
│                                                                                                                 │
│  The Consumer Protection E-commerce Guidelines, 2020, under Article 6, provides that the e-commerce platform    │
│  may offer a 'no questions asked return policy' and the buyer is not required to prove that the product was     │
│  defective.                                                                                                     │
│                                                                                                                 │
│  The Consumer Protection Act, 1986, is a pivotal legislation aimed at protecting the rights of consumers in     │
│  India. The Act, along with subsequent guidelines and amendments, provides a comprehensive framework for        │
│  consumer protection, including the right to seek refunds, replacements, or repairs for defective products or   │
│  services. This analysis delves into the legal implications of the Consumer Protection Act, 1986, and the       │
│  Consumer Protection E-commerce Guidelines, 2020, focusing on the strategic hierarchy of rules and applicable   │
│  laws in cases of conflict.                                                                                     │
│                                                       

Evaluation: PASS


╭──────────────────────────────────────────────── Tracing Status ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing is disabled.                                                                                     │
│                                                                                                                 │
│  To enable tracing, do any one of these:                                                                        │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

In [10]:
# Question 3
user_input = "A female colleague is being bothered by her boss at night over phone calls. Is there a law for women's safety at work?"
result = legal_crew.kickoff(inputs={"user_query": user_input})

# Verification Code
print(f"Evaluation: {'PASS' if 'POSH' in result.raw or 'Sexual Harassment' in result.raw else 'FAIL'}")

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name:                                                                                                          │
│  crew                                                                                                           │
│  ID:                                                                                                            │
│  f0593dc9-fdd5-4c1c-97e9-f38e700906b3                                                                           │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Search the database for all legal articles or sections relevant to: A female colleague is being          │
│  bothered by her boss at night over phone calls. Is there a law for women's safety at work?. Provide direct     │
│  quotes.                                                                                                        │
│  ID: 0e199204-0bca-45df-89b7-a700ad6f6c56                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Legal Researcher                                                                                        │
│                                                                                                                 │
│  Task: Search the database for all legal articles or sections relevant to: A female colleague is being          │
│  bothered by her boss at night over phone calls. Is there a law for women's safety at work?. Provide direct     │
│  quotes.                                                                                                        │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Legal Researcher                                                                                        │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  The previous analysis did not provide a complete picture of the legal provisions related to the question "Is   │
│  there a law for women's safety at work?" Therefore, further analysis is required.                              │
│                                                                                                                 │
│                                                                                                                 │
│  The Indian Penal Code, 1860, under Section 354 D, states that anyone found guilty of voyeurism shall be        │
│  punished with imprisonment for a term that may extend to 3 years and shall also be liable to fine:             │
│                                                                                                                 │
│                                                                                                                 │
│  "(1) The person who captures an image or video of another person without their consent in any private area,    │
│  for sexual gratification or for the purpose of blackmail, will be punished for voyeurism."                     │
│                                                                                                                 │
│                                                                                                                 │
│  The Indian Penal Code, 1860, under Section 354 D (2), further states:                                          │
│                                                                                                                 │
│                                                                                                                 │
│  "The person who captures an image or video of another person in a private area, without their permission, in   │
│  the context of blackmailing that other person by transmitting the images or videos to any person, shall be     │
│  punishable."                                                                                                   │
│                                                                                                                 │
│                                                                                                                 │
│  The Information Technology Act, 2000, under Section 66 E, defines as an "illicit traffic in girls" and         │
│  stipulates a punishment for any person who, in the course of providing or obtaining access to any computer     │
│  resource or communication service, captures or publishes any material in which a woman is involved in a        │
│  private setting without her consent, with the aim of creating a sexual sensation for himself or for any other  │
│  man.                                                                                                           │
│                                                                                                                 │
│                                                                                                                 │
│  Section 9 (1) of the Sexual Harassment of Women at Workplace (Prevention, Prohibition and Redressal) Bill of   │
│  2012 specifies:                                                                                                │
│                                                        

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name:                                                                                                          │
│  Search the database for all legal articles or sections relevant to: A female colleague is being bothered by    │
│  her boss at night over phone calls. Is there a law for women's safety at work?. Provide direct quotes.         │
│  Agent:                                                                                                         │
│  Legal Researcher                                                                                               │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭───────────────────────────────────────────────── ❌ LLM Error ──────────────────────────────────────────────────╮
│                                                                                                                 │
│  LLM Call Failed                                                                                                │
│  Error: litellm.RateLimitError: RateLimitError: GroqException - {"error":{"message":"Rate limit reached for     │
│  model `llama-3.3-70b-versatile` in organization `org_01kbn58qpqfjgstzyswyfty5g2` service tier `on_demand` on   │
│  tokens per minute (TPM): Limit 12000, Used 6652, Requested 10473. Please try again in 25.625s. Need more       │
│  tokens? Upgrade to Dev Tier today at                                                                           │
│  https://console.groq.com/settings/billing","type":"tokens","code":"rate_limit_exceeded"}}                      │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Failure ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Failed                                                                                                    │
│  Name:                                                                                                          │
│  Summarize the strategy into a final answer for the user. Be concise and professional.                          │
│  Agent:                                                                                                         │
│  Legal Advisor                                                                                                  │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────────── Crew Failure ──────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Failed                                                                                          │
│  Name:                                                                                                          │
│  crew                                                                                                           │
│  ID:                                                                                                            │
│  f0593dc9-fdd5-4c1c-97e9-f38e700906b3                                                                           │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Tracing Status ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing is disabled.                                                                                     │
│                                                                                                                 │
│  To enable tracing, do any one of these:                                                                        │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

RateLimitError: litellm.RateLimitError: RateLimitError: GroqException - {"error":{"message":"Rate limit reached for model `llama-3.3-70b-versatile` in organization `org_01kbn58qpqfjgstzyswyfty5g2` service tier `on_demand` on tokens per minute (TPM): Limit 12000, Used 6652, Requested 10473. Please try again in 25.625s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing","type":"tokens","code":"rate_limit_exceeded"}}


In [11]:
# Question 4
user_input = "Do daughters have the same right as sons to their father's house in India?"
result = legal_crew.kickoff(inputs={"user_query": user_input})

# Verification Code
print(f"Evaluation: {'PASS' if 'Succession' in result.raw or '2005' in result.raw else 'FAIL'}")

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name:                                                                                                          │
│  crew                                                                                                           │
│  ID:                                                                                                            │
│  f0593dc9-fdd5-4c1c-97e9-f38e700906b3                                                                           │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Search the database for all legal articles or sections relevant to: Do daughters have the same right as  │
│  sons to their father's house in India?. Provide direct quotes.                                                 │
│  ID: 0e199204-0bca-45df-89b7-a700ad6f6c56                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Legal Researcher                                                                                        │
│                                                                                                                 │
│  Task: Search the database for all legal articles or sections relevant to: Do daughters have the same right as  │
│  sons to their father's house in India?. Provide direct quotes.                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────── 🔧 Tool Execution Started (#5) ─────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: legal_research_tool                                                                                      │
│  Args: {'query': "Daughters' rights to property in India."}                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Tool legal_research_tool executed with result: ...


...

╭──────────────────────────────────────────────── 📋 Task Failure ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Failed                                                                                                    │
│  Name:                                                                                                          │
│  Search the database for all legal articles or sections relevant to: Do daughters have the same right as sons   │
│  to their father's house in India?. Provide direct quotes.                                                      │
│  Agent:                                                                                                         │
│  Legal Researcher                                                                                               │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

[CrewAIEventsBus] Warning: Event pairing mismatch. 'crew_kickoff_failed' closed 'task_started' (expected 
'crew_kickoff_started')

╭───────────────────────────────────────────────── Crew Failure ──────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Failed                                                                                          │
│  Name:                                                                                                          │
│  crew                                                                                                           │
│  ID:                                                                                                            │
│  f0593dc9-fdd5-4c1c-97e9-f38e700906b3                                                                           │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Tracing Status ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing is disabled.                                                                                     │
│                                                                                                                 │
│  To enable tracing, do any one of these:                                                                        │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

RateLimitError: litellm.RateLimitError: RateLimitError: GroqException - {"error":{"message":"Rate limit reached for model `llama-3.1-8b-instant` in organization `org_01kbn58qpqfjgstzyswyfty5g2` service tier `on_demand` on tokens per minute (TPM): Limit 6000, Used 3053, Requested 3166. Please try again in 2.19s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing","type":"tokens","code":"rate_limit_exceeded"}}


In [13]:
# Question 5
user_input = "Police searched phone/WhatsApp without a warrant for protest evidence. Does this violate Privacy laws?"
result = legal_crew.kickoff(inputs={"user_query": user_input})

# Verification Code
print(f"Evaluation: {'PASS' if 'Privacy' in result.raw or 'Puttaswamy' in result.raw else 'FAIL'}")
print("Check: Did it mention that even for national security, 'procedure established by law' must be followed?")

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name:                                                                                                          │
│  crew                                                                                                           │
│  ID:                                                                                                            │
│  f0593dc9-fdd5-4c1c-97e9-f38e700906b3                                                                           │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Search the database for all legal articles or sections relevant to: Police searched phone/WhatsApp       │
│  without a warrant for protest evidence. Does this violate Privacy laws?. Provide direct quotes.                │
│  ID: 0e199204-0bca-45df-89b7-a700ad6f6c56                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Legal Researcher                                                                                        │
│                                                                                                                 │
│  Task: Search the database for all legal articles or sections relevant to: Police searched phone/WhatsApp       │
│  without a warrant for protest evidence. Does this violate Privacy laws?. Provide direct quotes.                │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Legal Researcher                                                                                        │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  "The Information Technology (Amendment) Act, 2008, under Section 69, is an exception to the general rule that  │
│  requires a warrant for search of computers or communication devices. This provision gives the government the   │
│  power to intercept any message in the course of transmission through a computer or communication device if it  │
│  has been made to believe that such message involves the commission of a cognizable offence, or that the        │
│  message relates to a cognizable offence, and such interception is necessary for the purpose of public safety   │
│  or public order."                                                                                              │
│                                                                                                                 │
│                                                                                                                 │
│                                                                                                                 │
│  "The Information Technology (Amendment) Act, 2008, under Section 69B, empowers the Central or state            │
│  government to authorise the interception of any information generated, transmitted, received or stored in any  │
│  computer resource through any computer network. However, the authorisation is granted on the grounds of        │
│  national security, public order, or for the purpose of preventing the commission of a cognisable offence, or   │
│  for public health, or for the protection of the interests of India, and it is made to believe for reasons to   │
│  be recorded in writing by an officer, not below the rank of Joint Secretary to the Government of India or an   │
│  officer, not below the rank of Additional Director General of Police or Director General of Police, as the     │
│  case may be, that such interception or monitoring is necessary.                                                │
│                                                                                                                 │
│  "However, the information of such interception under clause (b) of sub-section (2) shall be recorded and       │
│  preserved by the Designated Agency for at least six months. It may be provided to the person concerned, on     │
│  request, or as the Central or State Government, as the case may be, may direct."                               │
│                                                                                                                 │
│                                                                                                                 │
│                                                                                                                 │
│  "Section 80 of the Information Technology Act states: "Nothing contained in sections 65 to 69 of the Act       │
│  shall authorise any officer of the Government to intercept any message, while its passing through a            │
│  telecommunications exchange or through any other means of communication exclusively for the purpose of         │
│  electronic surveillance, or cryptography, for the purpose of suppressing the rights and freedoms as set out    │
│  in article 19 of Part III and under section 25 of the IT Act (i.e. Section 25(2) of the IT Act) to access      │
│  computer resources for the purpose of surveillance in 

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name:                                                                                                          │
│  Search the database for all legal articles or sections relevant to: Police searched phone/WhatsApp without a   │
│  warrant for protest evidence. Does this violate Privacy laws?. Provide direct quotes.                          │
│  Agent:                                                                                                         │
│  Legal Researcher                                                                                               │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭───────────────────────────────────────────────── ❌ LLM Error ──────────────────────────────────────────────────╮
│                                                                                                                 │
│  LLM Call Failed                                                                                                │
│  Error: litellm.RateLimitError: RateLimitError: GroqException - {"error":{"message":"Request too large for      │
│  model `llama-3.3-70b-versatile` in organization `org_01kbn58qpqfjgstzyswyfty5g2` service tier `on_demand` on   │
│  tokens per minute (TPM): Limit 12000, Requested 12295, please reduce your message size and try again. Need     │
│  more tokens? Upgrade to Dev Tier today at                                                                      │
│  https://console.groq.com/settings/billing","type":"tokens","code":"rate_limit_exceeded"}}                      │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Failure ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Failed                                                                                                    │
│  Name:                                                                                                          │
│  Summarize the strategy into a final answer for the user. Be concise and professional.                          │
│  Agent:                                                                                                         │
│  Legal Advisor                                                                                                  │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────────── Crew Failure ──────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Failed                                                                                          │
│  Name:                                                                                                          │
│  crew                                                                                                           │
│  ID:                                                                                                            │
│  f0593dc9-fdd5-4c1c-97e9-f38e700906b3                                                                           │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Tracing Status ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing is disabled.                                                                                     │
│                                                                                                                 │
│  To enable tracing, do any one of these:                                                                        │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

RateLimitError: litellm.RateLimitError: RateLimitError: GroqException - {"error":{"message":"Request too large for model `llama-3.3-70b-versatile` in organization `org_01kbn58qpqfjgstzyswyfty5g2` service tier `on_demand` on tokens per minute (TPM): Limit 12000, Requested 12295, please reduce your message size and try again. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing","type":"tokens","code":"rate_limit_exceeded"}}


In [14]:
# Question 2
user_input = "Brother paralyzed due to wrong injection in a govt hospital. They claim charity immunity. Can we use Consumer Court?"
result = legal_crew.kickoff(inputs={"user_query": user_input})

# Verification Code
print(f"Evaluation: {'PASS' if 'Medical Negligence' in result.raw or 'Deficiency' in result.raw else 'FAIL'}")
print("Check: Did it mention that 'free' services in govt hospitals have specific nuances under the CP Act?")

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name:                                                                                                          │
│  crew                                                                                                           │
│  ID:                                                                                                            │
│  f0593dc9-fdd5-4c1c-97e9-f38e700906b3                                                                           │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Search the database for all legal articles or sections relevant to: Brother paralyzed due to wrong       │
│  injection in a govt hospital. They claim charity immunity. Can we use Consumer Court?. Provide direct quotes.  │
│  ID: 0e199204-0bca-45df-89b7-a700ad6f6c56                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Legal Researcher                                                                                        │
│                                                                                                                 │
│  Task: Search the database for all legal articles or sections relevant to: Brother paralyzed due to wrong       │
│  injection in a govt hospital. They claim charity immunity. Can we use Consumer Court?. Provide direct quotes.  │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────── 🔧 Tool Execution Started (#7) ─────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: legal_research_tool                                                                                      │
│  Args: {'query': 'Charity immunity and Consumer Courts.'}                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Tool legal_research_tool executed with result: ...


...

╭──────────────────────────────────────────────── 📋 Task Failure ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Failed                                                                                                    │
│  Name:                                                                                                          │
│  Search the database for all legal articles or sections relevant to: Brother paralyzed due to wrong injection   │
│  in a govt hospital. They claim charity immunity. Can we use Consumer Court?. Provide direct quotes.            │
│  Agent:                                                                                                         │
│  Legal Researcher                                                                                               │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

[CrewAIEventsBus] Warning: Event pairing mismatch. 'crew_kickoff_failed' closed 'task_started' (expected 
'crew_kickoff_started')

╭───────────────────────────────────────────────── Crew Failure ──────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Failed                                                                                          │
│  Name:                                                                                                          │
│  crew                                                                                                           │
│  ID:                                                                                                            │
│  f0593dc9-fdd5-4c1c-97e9-f38e700906b3                                                                           │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Tracing Status ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing is disabled.                                                                                     │
│                                                                                                                 │
│  To enable tracing, do any one of these:                                                                        │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

RateLimitError: litellm.RateLimitError: RateLimitError: GroqException - {"error":{"message":"Rate limit reached for model `llama-3.1-8b-instant` in organization `org_01kbn58qpqfjgstzyswyfty5g2` service tier `on_demand` on tokens per minute (TPM): Limit 6000, Used 4183, Requested 4278. Please try again in 24.61s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing","type":"tokens","code":"rate_limit_exceeded"}}
